<a href="https://colab.research.google.com/github/SANGHATI23/neurofhir-qc/blob/main/02_NeuroFHIR_QC_HAPI_Server_and_Read_Operations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NeuroFHIR-QC — Notebook 02
## HAPI FHIR Server Seeding and Verified Patient-Context Reads

**Project root:** `/content/drive/MyDrive/neurofhir-qc`  
**FHIR target:** R4  
**Default endpoint:** `https://hapi.fhir.org/baseR4`  
**Data policy:** synthetic FHIR records only

This notebook:

- retrieves and archives the server `CapabilityStatement`;
- seeds the 15 synthetic resources from Notebook 01 in one atomic transaction;
- reads all 15 resources back by `ResourceType/id`;
- reconstructs the stable, progression, and low-confidence patient contexts through FHIR searches;
- verifies critical-field preservation, chronology, and reference integrity;
- creates reusable FHIR client code and an execution audit.

**Boundary:** this notebook seeds synthetic source context only. It does not run MRI segmentation, create a current AI-derived result, calculate QC, perform human review, or execute the later AI-result write-back workflow.

**Public-server warning:** the default HAPI endpoint is a public test server. Never send PHI, confidential information, credentials, or real patient identifiers.

In [1]:
# Cell 1 — Mount Drive and enforce the Notebook 01 evidence gate

from __future__ import annotations

import csv
import hashlib
import json
import os
import re
import shutil
import textwrap
import time
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path
from time import perf_counter
from typing import Any
from urllib.parse import urlparse

import requests
import urllib3
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

try:
    from google.colab import drive
except ImportError as exc:
    raise RuntimeError("Open this notebook in Google Colab.") from exc

drive.mount("/content/drive")

PROJECT_ROOT = Path("/content/drive/MyDrive/neurofhir-qc")
CONFIG_PATH = PROJECT_ROOT / "project_config.json"
NOTEBOOK_MANIFEST_PATH = PROJECT_ROOT / "notebook_manifest.json"
NOTEBOOK_01_AUDIT_PATH = (
    PROJECT_ROOT / "evaluation/results/notebook_01_synthetic_fhir_audit.json"
)
NOTEBOOK_01_ROOT = PROJECT_ROOT / "data/synthetic_fhir/notebook_01"
RESOURCE_INDEX_PATH = NOTEBOOK_01_ROOT / "resource_index.json"
DEMO_CASE_MANIFEST_PATH = NOTEBOOK_01_ROOT / "demo_case_manifest.json"
NOTEBOOK_01_REFERENCE_PATH = NOTEBOOK_01_ROOT / "reference_integrity.json"

def load_json(path: Path) -> Any:
    with path.open("r", encoding="utf-8") as handle:
        return json.load(handle)

def write_json(path: Path, payload: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(path.suffix + ".tmp")
    with temp.open("w", encoding="utf-8") as handle:
        json.dump(payload, handle, indent=2, ensure_ascii=False)
        handle.write("\n")
    temp.replace(path)

def utc_now() -> str:
    return (
        datetime.now(timezone.utc)
        .replace(microsecond=0)
        .isoformat()
        .replace("+00:00", "Z")
    )

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

required_inputs = [
    CONFIG_PATH,
    NOTEBOOK_MANIFEST_PATH,
    NOTEBOOK_01_AUDIT_PATH,
    RESOURCE_INDEX_PATH,
    DEMO_CASE_MANIFEST_PATH,
    NOTEBOOK_01_REFERENCE_PATH,
]
missing = [
    str(path)
    for path in required_inputs
    if not path.exists() or path.stat().st_size == 0
]
if missing:
    raise FileNotFoundError(
        "Notebook 01 evidence is incomplete:\n"
        + "\n".join(f" - {path}" for path in missing)
    )

project_config = load_json(CONFIG_PATH)
notebook_manifest = load_json(NOTEBOOK_MANIFEST_PATH)
notebook_01_audit = load_json(NOTEBOOK_01_AUDIT_PATH)

def notebook_entries(manifest: Any) -> list[dict[str, Any]]:
    if isinstance(manifest, list):
        return manifest
    if isinstance(manifest, dict):
        for key in ("notebooks", "entries", "workflow"):
            value = manifest.get(key)
            if isinstance(value, list):
                return value
    raise ValueError("Unrecognized notebook_manifest.json structure.")

def normalize_number(value: Any) -> str:
    match = re.search(r"\d+", str(value))
    return match.group(0).zfill(2) if match else str(value)

def find_entry(number: str) -> dict[str, Any]:
    target = normalize_number(number)
    for entry in notebook_entries(notebook_manifest):
        candidates = [
            entry.get("number"),
            entry.get("notebook_number"),
            entry.get("id"),
            entry.get("filename"),
        ]
        if any(
            normalize_number(value) == target
            for value in candidates
            if value is not None
        ):
            return entry
    raise KeyError(f"Notebook {target} is missing from the manifest.")

notebook_01_entry = find_entry("01")
notebook_02_entry = find_entry("02")

if not notebook_01_audit.get("validation", {}).get("passed", False):
    raise RuntimeError("Notebook 01 validation did not pass.")

notebook_01_status = str(
    notebook_01_entry.get("status", notebook_01_audit.get("status", ""))
).lower()
accepted_statuses = {
    "completed",
    "complete",
    "passed",
    "executed_pending_notebook_save",
}
if notebook_01_status not in accepted_statuses:
    raise RuntimeError(
        f"Notebook 01 is not ready. Current status: {notebook_01_status!r}"
    )

NOTEBOOK_FILENAME = notebook_02_entry.get(
    "filename",
    "02_NeuroFHIR_QC_HAPI_Server_and_Read_Operations.ipynb",
)
NOTEBOOK_SAVE_PATH = PROJECT_ROOT / "notebooks" / NOTEBOOK_FILENAME

SERVER_ARTIFACT_ROOT = PROJECT_ROOT / "data/synthetic_fhir/notebook_02"
EVALUATION_ROOT = (
    PROJECT_ROOT / "evaluation/results/notebook_02_hapi_server_reads"
)
DIRECT_READ_ROOT = EVALUATION_ROOT / "direct_reads"
SEARCH_ROOT = EVALUATION_ROOT / "search_bundles"
CASE_CONTEXT_ROOT = SERVER_ARTIFACT_ROOT / "case_contexts"

for directory in (
    SERVER_ARTIFACT_ROOT,
    EVALUATION_ROOT,
    DIRECT_READ_ROOT,
    SEARCH_ROOT,
    CASE_CONTEXT_ROOT,
):
    directory.mkdir(parents=True, exist_ok=True)

print("=" * 90)
print("✅ Notebook 01 evidence gate passed")
print(f"✅ Notebook 01 status: {notebook_01_status}")
print(f"📓 Notebook 02 target path: {NOTEBOOK_SAVE_PATH}")
print("=" * 90)

Mounted at /content/drive
✅ Notebook 01 evidence gate passed
✅ Notebook 01 status: executed_pending_notebook_save
📓 Notebook 02 target path: /content/drive/MyDrive/neurofhir-qc/notebooks/02_NeuroFHIR_QC_HAPI_Server_and_Read_Operations.ipynb


In [2]:
# Cell 2 — Load and revalidate all 15 synthetic resources

resource_index = load_json(RESOURCE_INDEX_PATH)
demo_case_manifest = load_json(DEMO_CASE_MANIFEST_PATH)
reference_report_01 = load_json(NOTEBOOK_01_REFERENCE_PATH)

rows = resource_index.get("resources", [])
if len(rows) != 15:
    raise AssertionError(f"Expected 15 resources, found {len(rows)}.")

local_resources: list[dict[str, Any]] = []
local_by_reference: dict[str, dict[str, Any]] = {}

for row in rows:
    path = PROJECT_ROOT / row["relative_path"]
    resource = load_json(path)
    reference = f"{resource['resourceType']}/{resource['id']}"

    if reference != row["reference"]:
        raise AssertionError(f"Index mismatch for {path}.")
    if reference in local_by_reference:
        raise AssertionError(f"Duplicate reference: {reference}")

    tags = resource.get("meta", {}).get("tag", [])
    synthetic = any(
        isinstance(tag, dict)
        and tag.get("code") == "synthetic"
        and "neurofhir-qc.example.org" in str(tag.get("system", ""))
        for tag in tags
    )
    if not synthetic:
        raise AssertionError(f"{reference} lacks the synthetic tag.")

    local_resources.append(resource)
    local_by_reference[reference] = resource

counts = Counter(resource["resourceType"] for resource in local_resources)
expected_counts = {
    "Patient": 3,
    "Condition": 3,
    "ImagingStudy": 6,
    "Observation": 3,
}
if dict(counts) != expected_counts:
    raise AssertionError(f"Unexpected resource counts: {dict(counts)}")

if reference_report_01.get("resolved_reference_count") != reference_report_01.get(
    "reference_count"
):
    raise AssertionError("Notebook 01 reference integrity was not complete.")

case_entries = demo_case_manifest.get("cases", [])
if {case["case_id"] for case in case_entries} != {
    "stable",
    "progression",
    "low-confidence",
}:
    raise AssertionError("The three locked cases are not present.")

for observation in [
    resource
    for resource in local_resources
    if resource["resourceType"] == "Observation"
]:
    if observation.get("status") != "final":
        raise AssertionError("Historical prior Observations must be final.")

print("=" * 90)
print("✅ Loaded 15 synthetic FHIR resources")
for resource_type in ("Patient", "Condition", "ImagingStudy", "Observation"):
    print(f"   {resource_type}: {counts[resource_type]}")
print("✅ Synthetic tags and Notebook 01 reference integrity revalidated")
print("=" * 90)

✅ Loaded 15 synthetic FHIR resources
   Patient: 3
   Condition: 3
   ImagingStudy: 6
   Observation: 3
✅ Synthetic tags and Notebook 01 reference integrity revalidated


In [3]:
# Cell 3 — Configure the FHIR client and network log

FHIR_BASE_URL = os.getenv(
    "HAPI_FHIR_BASE_URL",
    "https://hapi.fhir.org/baseR4",
).strip().rstrip("/")
REQUEST_TIMEOUT_SECONDS = int(
    os.getenv("FHIR_REQUEST_TIMEOUT_SECONDS", "90")
)

ALLOW_SYNTHETIC_CONTEXT_SEEDING = True
SYNTHETIC_ONLY_ACKNOWLEDGED = True

parsed_url = urlparse(FHIR_BASE_URL)
if parsed_url.scheme not in {"http", "https"} or not parsed_url.netloc:
    raise ValueError("HAPI_FHIR_BASE_URL must be a valid HTTP(S) URL.")
if not ALLOW_SYNTHETIC_CONTEXT_SEEDING:
    raise RuntimeError("Synthetic context seeding must be explicitly enabled.")
if not SYNTHETIC_ONLY_ACKNOWLEDGED:
    raise RuntimeError("Synthetic-only acknowledgement is required.")

session = requests.Session()
session.headers.update(
    {
        "Accept": "application/fhir+json",
        "Content-Type": "application/fhir+json",
        "User-Agent": (
            f"NeuroFHIR-QC/{project_config.get('version', '0.1.0')} "
            "Notebook-02"
        ),
    }
)
retry = Retry(
    total=4,
    connect=4,
    read=4,
    status=4,
    backoff_factor=1.0,
    status_forcelist=(429, 500, 502, 503, 504),
    allowed_methods=frozenset({"GET", "HEAD", "OPTIONS"}),
    respect_retry_after_header=True,
    raise_on_status=False,
)
adapter = HTTPAdapter(max_retries=retry)
session.mount("https://", adapter)
session.mount("http://", adapter)

network_log: list[dict[str, Any]] = []

def outcome_text(payload: Any) -> str:
    if not isinstance(payload, dict) or payload.get("resourceType") != "OperationOutcome":
        return ""
    messages = []
    for issue in payload.get("issue", []):
        if not isinstance(issue, dict):
            continue
        details = issue.get("details", {})
        messages.append(
            " | ".join(
                value
                for value in (
                    str(issue.get("severity", "")),
                    str(details.get("text", "")) if isinstance(details, dict) else "",
                    str(issue.get("diagnostics", "")),
                )
                if value
            )
        )
    return " || ".join(messages[:5])

def fhir_request(
    method: str,
    path: str = "",
    *,
    params: dict[str, Any] | None = None,
    json_body: dict[str, Any] | None = None,
    expected_statuses: set[int] | None = None,
    purpose: str,
) -> tuple[dict[str, Any], requests.Response]:
    url = FHIR_BASE_URL if not path else f"{FHIR_BASE_URL}/{path.lstrip('/')}"
    started = perf_counter()
    response = session.request(
        method=method.upper(),
        url=url,
        params=params,
        json=json_body,
        timeout=(15, REQUEST_TIMEOUT_SECONDS),
    )
    elapsed = round(perf_counter() - started, 4)

    try:
        payload = response.json()
    except ValueError:
        payload = {
            "resourceType": "NonFHIRResponse",
            "text": response.text[:2000],
        }

    network_log.append(
        {
            "timestamp_utc": utc_now(),
            "purpose": purpose,
            "method": method.upper(),
            "url": response.url,
            "status_code": response.status_code,
            "elapsed_seconds": elapsed,
            "response_resource_type": (
                payload.get("resourceType")
                if isinstance(payload, dict)
                else None
            ),
        }
    )

    allowed = expected_statuses or set(range(200, 300))
    if response.status_code not in allowed:
        raise RuntimeError(
            f"{purpose} failed: {response.status_code} {response.reason}. "
            f"{outcome_text(payload) or str(payload)[:1500]}"
        )
    if not isinstance(payload, dict):
        raise RuntimeError(f"{purpose} returned a non-object JSON response.")
    return payload, response

print("=" * 90)
print("✅ FHIR client configured")
print(f"🌐 Server: {FHIR_BASE_URL}")
if parsed_url.netloc.lower() == "hapi.fhir.org":
    print("⚠️ Public HAPI test server selected; data are visible and temporary")
print("=" * 90)

✅ FHIR client configured
🌐 Server: https://hapi.fhir.org/baseR4
⚠️ Public HAPI test server selected; data are visible and temporary


In [4]:
# Cell 4 — Retrieve and audit the CapabilityStatement

CAPABILITY_PATH = EVALUATION_ROOT / "capability_statement.json"
CAPABILITY_SUMMARY_PATH = EVALUATION_ROOT / "capability_summary.json"

capability, capability_http = fhir_request(
    "GET",
    "metadata",
    expected_statuses={200},
    purpose="retrieve CapabilityStatement",
)

if capability.get("resourceType") != "CapabilityStatement":
    raise AssertionError("The metadata endpoint did not return CapabilityStatement.")

server_fhir_version = str(capability.get("fhirVersion", ""))
if not server_fhir_version.startswith("4.0"):
    raise AssertionError(
        f"FHIR R4 is required; server reported {server_fhir_version!r}."
    )

advertised: dict[str, set[str]] = {}
for rest in capability.get("rest", []):
    if not isinstance(rest, dict):
        continue
    for definition in rest.get("resource", []):
        if not isinstance(definition, dict):
            continue
        resource_type = definition.get("type")
        interactions = {
            item.get("code")
            for item in definition.get("interaction", [])
            if isinstance(item, dict) and item.get("code")
        }
        if resource_type:
            advertised.setdefault(str(resource_type), set()).update(interactions)

required_types = {"Patient", "Condition", "ImagingStudy", "Observation"}
missing_types = sorted(required_types - set(advertised))
if missing_types:
    raise AssertionError(f"Server does not advertise: {missing_types}")

required_interactions = {"read", "search-type", "update"}
gaps = {
    resource_type: sorted(required_interactions - advertised[resource_type])
    for resource_type in sorted(required_types)
    if required_interactions - advertised[resource_type]
}
if gaps:
    raise AssertionError(f"Missing required interactions: {gaps}")

software = capability.get("software", {})
capability_summary = {
    "retrieved_utc": utc_now(),
    "fhir_base_url": FHIR_BASE_URL,
    "fhir_version": server_fhir_version,
    "http_status": capability_http.status_code,
    "software_name": software.get("name") if isinstance(software, dict) else None,
    "software_version": software.get("version") if isinstance(software, dict) else None,
    "required_resources_present": True,
    "required_interactions_present": True,
    "interactions": {
        resource_type: sorted(advertised[resource_type])
        for resource_type in sorted(required_types)
    },
}

write_json(CAPABILITY_PATH, capability)
write_json(CAPABILITY_SUMMARY_PATH, capability_summary)

print("=" * 90)
print("✅ CapabilityStatement retrieved and archived")
print(f"✅ Server FHIR version: {server_fhir_version}")
print("✅ Required resources and read/search/update interactions advertised")
print(
    f"🖥️ Server software: {capability_summary['software_name']} "
    f"{capability_summary['software_version']}"
)
print("=" * 90)

✅ CapabilityStatement retrieved and archived
✅ Server FHIR version: 4.0.1
✅ Required resources and read/search/update interactions advertised
🖥️ Server software: HAPI FHIR Server 8.11.16-SNAPSHOT/7e7129efb5/2026-07-08


In [5]:
# Cell 5 — Seed all 15 resources in one atomic transaction

TRANSACTION_REQUEST_PATH = (
    SERVER_ARTIFACT_ROOT / "seed_transaction_request.json"
)
TRANSACTION_RESPONSE_PATH = (
    EVALUATION_ROOT / "seed_transaction_response.json"
)
TRANSACTION_STATUS_PATH = (
    EVALUATION_ROOT / "transaction_entry_statuses.json"
)
TRANSACTION_STATUS_CSV_PATH = (
    EVALUATION_ROOT / "transaction_entry_statuses.csv"
)

order = {"Patient": 0, "Condition": 1, "ImagingStudy": 2, "Observation": 3}
ordered_resources = sorted(
    local_resources,
    key=lambda resource: (order[resource["resourceType"]], resource["id"]),
)

transaction_bundle = {
    "resourceType": "Bundle",
    "type": "transaction",
    "timestamp": utc_now(),
    "identifier": {
        "system": (
            "https://neurofhir-qc.example.org/fhir/"
            "NamingSystem/notebook-transaction"
        ),
        "value": "notebook-02-synthetic-context-seed",
    },
    "entry": [
        {
            "fullUrl": (
                f"{FHIR_BASE_URL}/{resource['resourceType']}/{resource['id']}"
            ),
            "resource": resource,
            "request": {
                "method": "PUT",
                "url": f"{resource['resourceType']}/{resource['id']}",
            },
        }
        for resource in ordered_resources
    ],
}

if len(transaction_bundle["entry"]) != 15:
    raise AssertionError("The transaction must contain 15 entries.")

write_json(TRANSACTION_REQUEST_PATH, transaction_bundle)

transaction_response, transaction_http = fhir_request(
    "POST",
    "",
    json_body=transaction_bundle,
    expected_statuses={200},
    purpose="seed 15 synthetic resources",
)

if transaction_response.get("resourceType") != "Bundle":
    raise AssertionError("Transaction response is not a Bundle.")
if transaction_response.get("type") != "transaction-response":
    raise AssertionError("Expected Bundle.type=transaction-response.")

response_entries = transaction_response.get("entry", [])
if len(response_entries) != 15:
    raise AssertionError(
        f"Expected 15 response entries, found {len(response_entries)}."
    )

status_rows = []
failures = []
for resource, response_entry in zip(ordered_resources, response_entries):
    response_data = response_entry.get("response", {})
    status_text = str(response_data.get("status", ""))
    match = re.match(r"^(\d{3})", status_text)
    status_code = int(match.group(1)) if match else None

    row = {
        "resource_type": resource["resourceType"],
        "resource_id": resource["id"],
        "request_method": "PUT",
        "response_status": status_text,
        "response_status_code": status_code,
        "response_location": response_data.get("location"),
        "response_etag": response_data.get("etag"),
        "response_last_modified": response_data.get("lastModified"),
    }
    status_rows.append(row)
    if status_code is None or not 200 <= status_code < 300:
        failures.append(row)

if failures:
    raise AssertionError(f"Transaction entry failures: {failures}")

write_json(TRANSACTION_RESPONSE_PATH, transaction_response)
write_json(
    TRANSACTION_STATUS_PATH,
    {
        "executed_utc": utc_now(),
        "fhir_base_url": FHIR_BASE_URL,
        "http_status": transaction_http.status_code,
        "successful_entry_count": len(status_rows),
        "failed_entry_count": 0,
        "entries": status_rows,
    },
)

with TRANSACTION_STATUS_CSV_PATH.open(
    "w",
    encoding="utf-8",
    newline="",
) as handle:
    writer = csv.DictWriter(handle, fieldnames=list(status_rows[0]))
    writer.writeheader()
    writer.writerows(status_rows)

transaction_success_rate = len(status_rows) / 15

print("=" * 90)
print("✅ Atomic transaction completed")
print(f"✅ Successful entries: {len(status_rows)}/15")
print(f"✅ Transaction success rate: {transaction_success_rate:.1%}")
print("✅ Deterministic PUT makes the seed rerunnable without duplicate ids")
print("=" * 90)

✅ Atomic transaction completed
✅ Successful entries: 15/15
✅ Transaction success rate: 100.0%
✅ Deterministic PUT makes the seed rerunnable without duplicate ids


In [6]:
# Cell 6 — Directly read all 15 resources and verify critical fields

def critical_signature(resource: dict[str, Any]) -> dict[str, Any]:
    resource_type = resource["resourceType"]
    signature: dict[str, Any] = {
        "resourceType": resource_type,
        "id": resource["id"],
    }

    if resource_type == "Patient":
        signature.update(
            {
                "active": resource.get("active"),
                "identifier": resource.get("identifier"),
                "name": resource.get("name"),
                "gender": resource.get("gender"),
                "birthDate": resource.get("birthDate"),
            }
        )
    elif resource_type == "Condition":
        signature.update(
            {
                "clinicalStatus": resource.get("clinicalStatus"),
                "verificationStatus": resource.get("verificationStatus"),
                "code": resource.get("code"),
                "subject": resource.get("subject"),
                "onsetDateTime": resource.get("onsetDateTime"),
            }
        )
    elif resource_type == "ImagingStudy":
        signature.update(
            {
                "identifier": resource.get("identifier"),
                "status": resource.get("status"),
                "modality": resource.get("modality"),
                "subject": resource.get("subject"),
                "started": resource.get("started"),
                "reasonReference": resource.get("reasonReference"),
                "numberOfSeries": resource.get("numberOfSeries"),
                "numberOfInstances": resource.get("numberOfInstances"),
                "series": resource.get("series"),
            }
        )
    elif resource_type == "Observation":
        signature.update(
            {
                "status": resource.get("status"),
                "code": resource.get("code"),
                "subject": resource.get("subject"),
                "focus": resource.get("focus"),
                "effectiveDateTime": resource.get("effectiveDateTime"),
                "valueQuantity": resource.get("valueQuantity"),
                "derivedFrom": resource.get("derivedFrom"),
            }
        )
    else:
        raise ValueError(f"Unsupported resource type: {resource_type}")

    return signature

server_by_reference: dict[str, dict[str, Any]] = {}
direct_rows = []
mismatches = []

for local_resource in ordered_resources:
    reference = (
        f"{local_resource['resourceType']}/{local_resource['id']}"
    )
    server_resource, response = fhir_request(
        "GET",
        reference,
        expected_statuses={200},
        purpose=f"direct read {reference}",
    )

    if server_resource.get("resourceType") != local_resource["resourceType"]:
        raise AssertionError(f"{reference}: wrong resourceType returned.")
    if server_resource.get("id") != local_resource["id"]:
        raise AssertionError(f"{reference}: wrong id returned.")

    matches = (
        critical_signature(local_resource)
        == critical_signature(server_resource)
    )
    if not matches:
        mismatches.append(reference)

    server_by_reference[reference] = server_resource
    output_path = (
        DIRECT_READ_ROOT
        / f"{local_resource['resourceType'].lower()}-{local_resource['id']}.json"
    )
    write_json(output_path, server_resource)

    direct_rows.append(
        {
            "reference": reference,
            "http_status": response.status_code,
            "critical_signature_matches": matches,
            "server_version_id": server_resource.get("meta", {}).get("versionId"),
            "server_last_updated": server_resource.get("meta", {}).get("lastUpdated"),
            "archived_relative_path": (
                output_path.relative_to(PROJECT_ROOT).as_posix()
            ),
        }
    )

if mismatches:
    raise AssertionError(
        "Critical fields changed during round-trip: "
        + ", ".join(mismatches)
    )

direct_read_success_rate = len(server_by_reference) / 15
signature_match_rate = (
    sum(1 for row in direct_rows if row["critical_signature_matches"])
    / len(direct_rows)
)

DIRECT_READ_REPORT_PATH = EVALUATION_ROOT / "direct_read_report.json"
write_json(
    DIRECT_READ_REPORT_PATH,
    {
        "executed_utc": utc_now(),
        "expected_count": 15,
        "successful_count": len(direct_rows),
        "direct_read_success_rate": direct_read_success_rate,
        "critical_signature_match_rate": signature_match_rate,
        "resources": direct_rows,
    },
)

print("=" * 90)
print(f"✅ Direct reads: {len(direct_rows)}/15")
print(f"✅ Direct-read success rate: {direct_read_success_rate:.1%}")
print(f"✅ Critical-field preservation: {signature_match_rate:.1%}")
print("=" * 90)

✅ Direct reads: 15/15
✅ Direct-read success rate: 100.0%
✅ Critical-field preservation: 100.0%


In [7]:
# Cell 7 — Reconstruct patient contexts through FHIR searches and verify references

def bundle_resources(bundle: dict[str, Any]) -> list[dict[str, Any]]:
    if bundle.get("resourceType") != "Bundle":
        raise AssertionError("Search did not return a Bundle.")
    if bundle.get("type") != "searchset":
        raise AssertionError("Search did not return Bundle.type=searchset.")
    return [
        entry["resource"]
        for entry in bundle.get("entry", [])
        if isinstance(entry, dict)
        and isinstance(entry.get("resource"), dict)
    ]

def search_until_found(
    resource_type: str,
    params: dict[str, Any],
    expected_ids: set[str],
    purpose: str,
    attempts: int = 6,
) -> tuple[dict[str, Any], set[str], int]:
    returned_ids: set[str] = set()
    last_bundle: dict[str, Any] | None = None

    for attempt in range(1, attempts + 1):
        bundle, _ = fhir_request(
            "GET",
            resource_type,
            params={**params, "_count": "50"},
            expected_statuses={200},
            purpose=f"{purpose} attempt {attempt}",
        )
        returned_ids = {
            str(resource["id"])
            for resource in bundle_resources(bundle)
            if resource.get("resourceType") == resource_type
            and resource.get("id")
        }
        last_bundle = bundle
        if expected_ids.issubset(returned_ids):
            return bundle, returned_ids, attempt
        if attempt < attempts:
            time.sleep(2)

    raise AssertionError(
        f"{purpose}: expected {sorted(expected_ids)}, "
        f"returned {sorted(returned_ids)}"
    )

search_results = []

for case in case_entries:
    case_id = case["case_id"]
    patient_reference = case["patient_reference"]
    patient_id = patient_reference.split("/", 1)[1]
    patient = local_by_reference[patient_reference]
    identifier = patient["identifier"][0]
    identifier_token = f"{identifier['system']}|{identifier['value']}"

    definitions = [
        (
            "patient_by_identifier",
            "Patient",
            {"identifier": identifier_token},
            {patient_id},
        ),
        (
            "conditions_by_subject",
            "Condition",
            {"subject": patient_reference},
            {case["condition_reference"].split("/", 1)[1]},
        ),
        (
            "imaging_studies_by_subject",
            "ImagingStudy",
            {"subject": patient_reference},
            {
                case["baseline_imaging_reference"].split("/", 1)[1],
                case["followup_imaging_reference"].split("/", 1)[1],
            },
        ),
        (
            "observations_by_subject",
            "Observation",
            {"subject": patient_reference},
            {case["prior_observation_reference"].split("/", 1)[1]},
        ),
    ]

    query_rows = []
    context_resources: dict[str, list[dict[str, Any]]] = {}

    for label, resource_type, params, expected_ids in definitions:
        bundle, returned_ids, attempts_used = search_until_found(
            resource_type,
            params,
            expected_ids,
            f"{case_id} {label}",
        )
        bundle_path = SEARCH_ROOT / f"{case_id}_{label}.json"
        write_json(bundle_path, bundle)

        matched = [
            resource
            for resource in bundle_resources(bundle)
            if resource.get("id") in expected_ids
        ]
        context_resources[resource_type] = matched
        query_rows.append(
            {
                "label": label,
                "resource_type": resource_type,
                "parameters": params,
                "expected_ids": sorted(expected_ids),
                "returned_ids": sorted(returned_ids),
                "expected_ids_found": True,
                "attempts_used": attempts_used,
                "archived_bundle": (
                    bundle_path.relative_to(PROJECT_ROOT).as_posix()
                ),
            }
        )

    context_path = CASE_CONTEXT_ROOT / f"{case_id}_server_context.json"
    write_json(
        context_path,
        {
            "case_id": case_id,
            "retrieved_utc": utc_now(),
            "fhir_base_url": FHIR_BASE_URL,
            "patient_reference": patient_reference,
            "queries": query_rows,
            "resources": context_resources,
        },
    )

    search_results.append(
        {
            "case_id": case_id,
            "query_count": len(query_rows),
            "all_expected_ids_found": all(
                row["expected_ids_found"] for row in query_rows
            ),
            "context_relative_path": (
                context_path.relative_to(PROJECT_ROOT).as_posix()
            ),
            "queries": query_rows,
        }
    )

patient_context_success_rate = (
    sum(1 for result in search_results if result["all_expected_ids_found"])
    / len(search_results)
)
search_query_count = sum(result["query_count"] for result in search_results)

PATIENT_CONTEXT_REPORT_PATH = (
    EVALUATION_ROOT / "patient_context_search_report.json"
)
write_json(
    PATIENT_CONTEXT_REPORT_PATH,
    {
        "executed_utc": utc_now(),
        "case_count": len(search_results),
        "query_count": search_query_count,
        "successful_case_count": sum(
            1 for result in search_results if result["all_expected_ids_found"]
        ),
        "patient_context_success_rate": patient_context_success_rate,
        "cases": search_results,
    },
)

def collect_references(value: Any) -> list[str]:
    references = []
    if isinstance(value, dict):
        for key, nested in value.items():
            if key == "reference" and isinstance(nested, str):
                references.append(nested)
            else:
                references.extend(collect_references(nested))
    elif isinstance(value, list):
        for item in value:
            references.extend(collect_references(item))
    return references

reference_rows = []
unresolved = []
for source, resource in server_by_reference.items():
    for target in collect_references(resource):
        resolved = target in server_by_reference
        reference_rows.append(
            {
                "source_reference": source,
                "target_reference": target,
                "resolved_within_seeded_context": resolved,
            }
        )
        if not resolved:
            unresolved.append(f"{source} -> {target}")

if unresolved:
    raise AssertionError(
        "Unresolved server references:\n"
        + "\n".join(f" - {item}" for item in unresolved)
    )

for case in case_entries:
    baseline = server_by_reference[case["baseline_imaging_reference"]]
    followup = server_by_reference[case["followup_imaging_reference"]]
    observation = server_by_reference[case["prior_observation_reference"]]

    baseline_time = datetime.fromisoformat(
        baseline["started"].replace("Z", "+00:00")
    )
    followup_time = datetime.fromisoformat(
        followup["started"].replace("Z", "+00:00")
    )
    if not baseline_time < followup_time:
        raise AssertionError(
            f"{case['case_id']}: invalid baseline/follow-up chronology."
        )

    derived_from = {
        item.get("reference")
        for item in observation.get("derivedFrom", [])
        if isinstance(item, dict)
    }
    if case["baseline_imaging_reference"] not in derived_from:
        raise AssertionError(
            f"{case['case_id']}: prior Observation is not linked to baseline."
        )

resolved_reference_count = sum(
    1 for row in reference_rows if row["resolved_within_seeded_context"]
)
reference_integrity_rate = (
    resolved_reference_count / len(reference_rows)
    if reference_rows
    else 1.0
)

SERVER_REFERENCE_REPORT_PATH = (
    EVALUATION_ROOT / "server_reference_integrity_report.json"
)
write_json(
    SERVER_REFERENCE_REPORT_PATH,
    {
        "executed_utc": utc_now(),
        "reference_count": len(reference_rows),
        "resolved_reference_count": resolved_reference_count,
        "reference_integrity_rate": reference_integrity_rate,
        "references": reference_rows,
    },
)

print("=" * 90)
print("✅ All three patient contexts reconstructed through FHIR searches")
print(f"✅ Verified search queries: {search_query_count}")
print(f"✅ Patient-context success rate: {patient_context_success_rate:.1%}")
print(
    f"✅ Server reference integrity: "
    f"{resolved_reference_count}/{len(reference_rows)} "
    f"({reference_integrity_rate:.1%})"
)
print("✅ Baseline/follow-up chronology and prior-result linkage verified")
print("=" * 90)

✅ All three patient contexts reconstructed through FHIR searches
✅ Verified search queries: 12
✅ Patient-context success rate: 100.0%
✅ Server reference integrity: 24/24 (100.0%)
✅ Baseline/follow-up chronology and prior-result linkage verified


In [8]:
# Cell 8 — Create reusable FHIR client code and dependency documentation

FHIR_CLIENT_PATH = PROJECT_ROOT / "backend/app/services/fhir_client.py"
SEED_SCRIPT_PATH = PROJECT_ROOT / "scripts/seed_hapi.py"
REQUIREMENTS_PATH = PROJECT_ROOT / "requirements/fhir.txt"
DOCUMENTATION_PATH = (
    PROJECT_ROOT / "docs/HAPI_SERVER_READ_OPERATIONS.md"
)

fhir_client_source = """
# Reusable FHIR R4 JSON client for NeuroFHIR-QC.

from __future__ import annotations

from dataclasses import dataclass
from typing import Any

import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry


class FHIRClientError(RuntimeError):
    pass


@dataclass(frozen=True)
class FHIRClientConfig:
    base_url: str
    timeout_seconds: int = 60
    user_agent: str = "NeuroFHIR-QC/0.1.0"


class FHIRClient:
    def __init__(self, config: FHIRClientConfig) -> None:
        self.config = config
        self.base_url = config.base_url.rstrip("/")
        self.session = requests.Session()
        self.session.headers.update(
            {
                "Accept": "application/fhir+json",
                "Content-Type": "application/fhir+json",
                "User-Agent": config.user_agent,
            }
        )
        retry = Retry(
            total=3,
            connect=3,
            read=3,
            status=3,
            backoff_factor=1.0,
            status_forcelist=(429, 500, 502, 503, 504),
            allowed_methods=frozenset({"GET", "HEAD", "OPTIONS"}),
            respect_retry_after_header=True,
            raise_on_status=False,
        )
        adapter = HTTPAdapter(max_retries=retry)
        self.session.mount("https://", adapter)
        self.session.mount("http://", adapter)

    def _url(self, path: str = "") -> str:
        path = path.lstrip("/")
        return self.base_url if not path else f"{self.base_url}/{path}"

    def _request(
        self,
        method: str,
        path: str = "",
        *,
        params: dict[str, Any] | None = None,
        json_body: dict[str, Any] | None = None,
    ) -> dict[str, Any]:
        response = self.session.request(
            method=method.upper(),
            url=self._url(path),
            params=params,
            json=json_body,
            timeout=(15, self.config.timeout_seconds),
        )
        try:
            payload = response.json()
        except ValueError as exc:
            raise FHIRClientError(
                f"Server returned non-JSON content: {response.text[:1000]}"
            ) from exc

        if not 200 <= response.status_code < 300:
            raise FHIRClientError(
                f"{method.upper()} {response.url} failed with "
                f"{response.status_code}: {payload}"
            )
        if not isinstance(payload, dict):
            raise FHIRClientError("FHIR response must be a JSON object.")
        return payload

    def capabilities(self) -> dict[str, Any]:
        return self._request("GET", "metadata")

    def read(self, resource_type: str, resource_id: str) -> dict[str, Any]:
        return self._request("GET", f"{resource_type}/{resource_id}")

    def search(
        self,
        resource_type: str,
        **parameters: Any,
    ) -> dict[str, Any]:
        return self._request("GET", resource_type, params=parameters)

    def transaction(self, bundle: dict[str, Any]) -> dict[str, Any]:
        if bundle.get("resourceType") != "Bundle":
            raise ValueError("transaction() requires a FHIR Bundle.")
        if bundle.get("type") != "transaction":
            raise ValueError("Bundle.type must be transaction.")
        return self._request("POST", "", json_body=bundle)
"""

seed_script_source = """
# Seed NeuroFHIR-QC synthetic Notebook 01 context into a FHIR R4 server.

from __future__ import annotations

import json
import os
from pathlib import Path
from typing import Any

from backend.app.services.fhir_client import (
    FHIRClient,
    FHIRClientConfig,
)


PROJECT_ROOT = Path(
    os.getenv(
        "NEUROFHIR_QC_PROJECT_ROOT",
        "/content/drive/MyDrive/neurofhir-qc",
    )
)
RESOURCE_INDEX_PATH = (
    PROJECT_ROOT
    / "data/synthetic_fhir/notebook_01/resource_index.json"
)


def load_json(path: Path) -> Any:
    with path.open("r", encoding="utf-8") as handle:
        return json.load(handle)


def main() -> None:
    if (
        os.getenv("ALLOW_SYNTHETIC_CONTEXT_SEEDING", "false").lower()
        != "true"
    ):
        raise RuntimeError(
            "Set ALLOW_SYNTHETIC_CONTEXT_SEEDING=true. Never use PHI."
        )

    base_url = os.getenv(
        "HAPI_FHIR_BASE_URL",
        "https://hapi.fhir.org/baseR4",
    )
    index = load_json(RESOURCE_INDEX_PATH)
    resources = [
        load_json(PROJECT_ROOT / row["relative_path"])
        for row in index["resources"]
    ]

    bundle = {
        "resourceType": "Bundle",
        "type": "transaction",
        "entry": [
            {
                "fullUrl": (
                    f"{base_url.rstrip('/')}/"
                    f"{resource['resourceType']}/{resource['id']}"
                ),
                "resource": resource,
                "request": {
                    "method": "PUT",
                    "url": (
                        f"{resource['resourceType']}/{resource['id']}"
                    ),
                },
            }
            for resource in resources
        ],
    }

    client = FHIRClient(FHIRClientConfig(base_url=base_url))
    response = client.transaction(bundle)
    print(
        json.dumps(
            {
                "resource_count": len(resources),
                "response_type": response.get("type"),
                "statuses": [
                    entry.get("response", {}).get("status")
                    for entry in response.get("entry", [])
                ],
            },
            indent=2,
        )
    )


if __name__ == "__main__":
    main()
"""

documentation = f"""
# HAPI FHIR Server and Read Operations

Generated by `{NOTEBOOK_FILENAME}`.

## Demonstrated workflow

1. Retrieve the server CapabilityStatement.
2. Confirm FHIR R4 and required interactions.
3. Seed the 15 synthetic resources in one transaction.
4. Read every resource directly.
5. Reconstruct each patient context through searches.
6. Archive transaction, search, timing, and integrity evidence.

## Safety

The default endpoint is a public HAPI test server. Never send PHI,
confidential data, credentials, or real patient identifiers.

## Reseed from the command line

```bash
export ALLOW_SYNTHETIC_CONTEXT_SEEDING=true
export HAPI_FHIR_BASE_URL={FHIR_BASE_URL}
python scripts/seed_hapi.py
```

Deterministic PUT requests make the transaction idempotent.

## Scope boundary

This is synthetic source-context seeding. It is not AI-result write-back.
"""

FHIR_CLIENT_PATH.parent.mkdir(parents=True, exist_ok=True)
SEED_SCRIPT_PATH.parent.mkdir(parents=True, exist_ok=True)
REQUIREMENTS_PATH.parent.mkdir(parents=True, exist_ok=True)
DOCUMENTATION_PATH.parent.mkdir(parents=True, exist_ok=True)

FHIR_CLIENT_PATH.write_text(
    textwrap.dedent(fhir_client_source).strip() + "\n",
    encoding="utf-8",
)
SEED_SCRIPT_PATH.write_text(
    textwrap.dedent(seed_script_source).strip() + "\n",
    encoding="utf-8",
)
REQUIREMENTS_PATH.write_text(
    f"requests=={requests.__version__}\n"
    f"urllib3=={urllib3.__version__}\n",
    encoding="utf-8",
)
DOCUMENTATION_PATH.write_text(
    textwrap.dedent(documentation).strip() + "\n",
    encoding="utf-8",
)

compile(
    FHIR_CLIENT_PATH.read_text(encoding="utf-8"),
    str(FHIR_CLIENT_PATH),
    "exec",
)
compile(
    SEED_SCRIPT_PATH.read_text(encoding="utf-8"),
    str(SEED_SCRIPT_PATH),
    "exec",
)

reusable_files = [
    FHIR_CLIENT_PATH,
    SEED_SCRIPT_PATH,
    REQUIREMENTS_PATH,
    DOCUMENTATION_PATH,
]
empty = [
    str(path)
    for path in reusable_files
    if not path.exists() or path.stat().st_size == 0
]
if empty:
    raise FileNotFoundError(f"Reusable files are missing: {empty}")

print("=" * 90)
print("✅ Reusable FHIR client and seeding script created")
print("✅ Generated Python files passed syntax compilation")
print(
    f"✅ Dependencies recorded: requests {requests.__version__}, "
    f"urllib3 {urllib3.__version__}"
)
print("=" * 90)

✅ Reusable FHIR client and seeding script created
✅ Generated Python files passed syntax compilation
✅ Dependencies recorded: requests 2.32.4, urllib3 2.5.0


In [9]:
# Cell 9 — Create the final Notebook 02 audit and update the manifest

NETWORK_LOG_PATH = EVALUATION_ROOT / "network_request_log.json"
AUDIT_JSON_PATH = (
    PROJECT_ROOT
    / "evaluation/results/notebook_02_hapi_server_read_audit.json"
)
AUDIT_MD_PATH = (
    PROJECT_ROOT
    / "docs/NOTEBOOK_02_HAPI_SERVER_AND_READ_OPERATIONS.md"
)

write_json(
    NETWORK_LOG_PATH,
    {
        "generated_utc": utc_now(),
        "fhir_base_url": FHIR_BASE_URL,
        "request_count": len(network_log),
        "requests": network_log,
    },
)

required_metrics = {
    "transaction_success_rate": transaction_success_rate,
    "direct_read_success_rate": direct_read_success_rate,
    "critical_signature_match_rate": signature_match_rate,
    "patient_context_success_rate": patient_context_success_rate,
    "server_reference_integrity_rate": reference_integrity_rate,
}
failed_metrics = [
    name
    for name, value in required_metrics.items()
    if value != 1.0
]
if failed_metrics:
    raise AssertionError(
        f"Notebook 02 completion metrics failed: {failed_metrics}"
    )

core_files = [
    CAPABILITY_PATH,
    CAPABILITY_SUMMARY_PATH,
    TRANSACTION_REQUEST_PATH,
    TRANSACTION_RESPONSE_PATH,
    TRANSACTION_STATUS_PATH,
    TRANSACTION_STATUS_CSV_PATH,
    DIRECT_READ_REPORT_PATH,
    PATIENT_CONTEXT_REPORT_PATH,
    SERVER_REFERENCE_REPORT_PATH,
    NETWORK_LOG_PATH,
    FHIR_CLIENT_PATH,
    SEED_SCRIPT_PATH,
    REQUIREMENTS_PATH,
    DOCUMENTATION_PATH,
]
missing_core = [
    str(path)
    for path in core_files
    if not path.exists() or path.stat().st_size == 0
]
if missing_core:
    raise FileNotFoundError(
        "Notebook 02 evidence is missing:\n"
        + "\n".join(f" - {path}" for path in missing_core)
    )

NOTEBOOK_SAVE_PATH.parent.mkdir(parents=True, exist_ok=True)
candidates = [
    NOTEBOOK_SAVE_PATH,
    PROJECT_ROOT / NOTEBOOK_FILENAME,
    PROJECT_ROOT
    / "02_NeuroFHIR_QC_HAPI_Server_and_Read_Operations.ipynb",
]
candidates.extend(sorted(PROJECT_ROOT.glob("*02*.ipynb")))
candidates.extend(
    sorted((PROJECT_ROOT / "notebooks").glob("*02*.ipynb"))
)
found = next(
    (
        path
        for path in candidates
        if path.exists() and path.is_file() and path.stat().st_size > 0
    ),
    None,
)
if found and found.resolve() != NOTEBOOK_SAVE_PATH.resolve():
    shutil.copy2(found, NOTEBOOK_SAVE_PATH)

notebook_saved = (
    NOTEBOOK_SAVE_PATH.exists()
    and NOTEBOOK_SAVE_PATH.stat().st_size > 0
)
manifest_status = (
    "completed"
    if notebook_saved
    else "executed_pending_notebook_save"
)

checksum_paths = sorted(
    {
        path.resolve()
        for path in (
            core_files
            + list(DIRECT_READ_ROOT.glob("*.json"))
            + list(SEARCH_ROOT.glob("*.json"))
            + list(CASE_CONTEXT_ROOT.glob("*.json"))
        )
        if path.exists() and path.is_file()
    },
    key=lambda path: str(path),
)
checksums = [
    {
        "relative_path": path.relative_to(PROJECT_ROOT).as_posix(),
        "size_bytes": path.stat().st_size,
        "sha256": sha256_file(path),
    }
    for path in checksum_paths
]

metrics = {
    "server_fhir_version": server_fhir_version,
    "transaction_entries_successful": len(status_rows),
    "transaction_success_rate": transaction_success_rate,
    "direct_reads_successful": len(direct_rows),
    "direct_read_success_rate": direct_read_success_rate,
    "critical_signature_match_rate": signature_match_rate,
    "patient_context_cases_successful": 3,
    "patient_context_query_count": search_query_count,
    "patient_context_success_rate": patient_context_success_rate,
    "server_reference_count": len(reference_rows),
    "resolved_server_reference_count": resolved_reference_count,
    "server_reference_integrity_rate": reference_integrity_rate,
    "network_request_count": len(network_log),
}

audit = {
    "project_name": project_config["project_name"],
    "project_version": project_config.get("version", "0.1.0"),
    "notebook_number": "02",
    "notebook_filename": NOTEBOOK_FILENAME,
    "status": manifest_status,
    "audited_utc": utc_now(),
    "notebook_saved_to_project": notebook_saved,
    "required_notebook_save_path": str(NOTEBOOK_SAVE_PATH),
    "fhir_base_url": FHIR_BASE_URL,
    "server_type": (
        "public-hapi-test-server"
        if parsed_url.netloc.lower() == "hapi.fhir.org"
        else "configured-fhir-server"
    ),
    "data_policy": {
        "synthetic_fhir_only": True,
        "real_patient_data_allowed": False,
        "confidential_data_allowed": False,
    },
    "scope": {
        "source_context_seeded": True,
        "current_ai_observation_created": False,
        "segmentation_run": False,
        "qc_calculated": False,
        "human_review_transition_executed": False,
        "ai_result_writeback_performed": False,
    },
    "metrics": metrics,
    "checksum_inventory": checksums,
    "next_notebook": (
        "03 — Imaging Data Preparation, only after Notebook 02 "
        "is marked completed."
    ),
}
write_json(AUDIT_JSON_PATH, audit)

audit_markdown = f"""
# Notebook 02 — HAPI FHIR Server and Read Operations

**Notebook:** `{NOTEBOOK_FILENAME}`
**Status:** `{manifest_status}`
**FHIR server:** `{FHIR_BASE_URL}`
**FHIR version:** `{server_fhir_version}`
**Audited UTC:** {audit['audited_utc']}

## Evidence

| Metric | Result |
|---|---:|
| Successful transaction entries | {len(status_rows)}/15 |
| Transaction success rate | {transaction_success_rate:.1%} |
| Successful direct reads | {len(direct_rows)}/15 |
| Direct-read success rate | {direct_read_success_rate:.1%} |
| Critical-field preservation | {signature_match_rate:.1%} |
| Patient contexts reconstructed | 3/3 |
| Patient-context success rate | {patient_context_success_rate:.1%} |
| Server references resolved | {resolved_reference_count}/{len(reference_rows)} |
| Reference-integrity rate | {reference_integrity_rate:.1%} |
| Recorded HTTP requests | {len(network_log)} |

## Scope boundary

Notebook 02 seeded synthetic source context only. It did not run
segmentation, create a current AI result, calculate QC, perform human
review, or execute AI-result write-back.

## Completion gate

Required notebook path:

`{NOTEBOOK_SAVE_PATH}`

Notebook detected there: **{'yes' if notebook_saved else 'no'}**

If no, save this notebook there and rerun Cell 9.
"""

AUDIT_MD_PATH.parent.mkdir(parents=True, exist_ok=True)
AUDIT_MD_PATH.write_text(
    textwrap.dedent(audit_markdown).strip() + "\n",
    encoding="utf-8",
)

notebook_02_entry["status"] = manifest_status
notebook_02_entry["last_executed_utc"] = utc_now()
notebook_02_entry["server_base_url"] = FHIR_BASE_URL
notebook_02_entry["server_fhir_version"] = server_fhir_version
notebook_02_entry["seeded_resource_count"] = 15
notebook_02_entry["direct_read_success_rate"] = direct_read_success_rate
notebook_02_entry["patient_context_success_rate"] = (
    patient_context_success_rate
)
notebook_02_entry["audit_path"] = (
    AUDIT_JSON_PATH.relative_to(PROJECT_ROOT).as_posix()
)
write_json(NOTEBOOK_MANIFEST_PATH, notebook_manifest)

print("=" * 90)
print("✅ Notebook 02 execution evidence passed")
print("✅ 15/15 transaction entries succeeded")
print("✅ 15/15 direct reads succeeded")
print("✅ 3/3 patient contexts reconstructed")
print(f"✅ Audit JSON: {AUDIT_JSON_PATH}")
print(f"✅ Audit Markdown: {AUDIT_MD_PATH}")
print(f"📓 Manifest status: {manifest_status}")
if notebook_saved:
    print("🎯 Notebook 02 is complete; Notebook 03 may begin")
else:
    print("⚠️ Save this notebook into the required Drive path")
    print(f"   {NOTEBOOK_SAVE_PATH}")
    print("Then rerun Cell 9 before beginning Notebook 03.")
print("=" * 90)

✅ Notebook 02 execution evidence passed
✅ 15/15 transaction entries succeeded
✅ 15/15 direct reads succeeded
✅ 3/3 patient contexts reconstructed
✅ Audit JSON: /content/drive/MyDrive/neurofhir-qc/evaluation/results/notebook_02_hapi_server_read_audit.json
✅ Audit Markdown: /content/drive/MyDrive/neurofhir-qc/docs/NOTEBOOK_02_HAPI_SERVER_AND_READ_OPERATIONS.md
📓 Manifest status: executed_pending_notebook_save
⚠️ Save this notebook into the required Drive path
   /content/drive/MyDrive/neurofhir-qc/notebooks/02_NeuroFHIR_QC_HAPI_Server_and_Read_Operations.ipynb
Then rerun Cell 9 before beginning Notebook 03.
